In [1]:
import os
import random
import numpy as np
from ase.io import read, write
from ase import Atoms
from tqdm import tqdm

from dscribe.descriptors import SOAP

# ---------------------------
# Parameters
# ---------------------------
ref_path = "/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/meci/ethylene_main_meci/twist.xyz"
target_folder = "/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/"
output_folder = "./reconstructed_batch/"
os.makedirs(output_folder, exist_ok=True)

n_samples = 5  # number of molecules to reconstruct
max_steps = 2000
lr = 0.05
eps = 1e-4  # finite-difference epsilon
r_cut = 5.0
n_max = 8
l_max = 6
average = "inner"

# ---------------------------
# SOAP helper
# ---------------------------
def generate_SOAP(atoms, r_cut=5.0, n_max=8, l_max=6, average="inner"):
    species = list(set(atoms.get_chemical_symbols()))
    soap = SOAP(
        species=species,
        periodic=False,
        r_cut=r_cut,
        n_max=n_max,
        l_max=l_max,
        average=average,
        sparse=False
    )
    return soap.create([atoms]).flatten()

# ---------------------------
# Finite-difference gradient
# ---------------------------
def finite_gradient(current_geom, target_soap, eps=1e-4):
    grad = np.zeros_like(current_geom)
    for i in range(len(current_geom)):
        for j in range(3):
            pos = current_geom.copy()
            pos[i, j] += eps
            f_plus = generate_SOAP(Atoms(positions=pos, symbols=target_symbols))
            
            pos[i, j] -= 2 * eps
            f_minus = generate_SOAP(Atoms(positions=pos, symbols=target_symbols))
            
            grad[i, j] = (f_plus - f_minus).dot(target_soap) / (2 * eps)
    return grad

# ---------------------------
# Load reference
# ---------------------------
ref_geom = read(ref_path)
target_symbols = ref_geom.get_chemical_symbols()  # assume same species

# ---------------------------
# Sample target geometries
# ---------------------------
all_files = sorted([f for f in os.listdir(target_folder) if f.endswith(".xyz")])
sample_files = random.sample(all_files, min(n_samples, len(all_files)))

# ---------------------------
# Batch reconstruction
# ---------------------------
for target_file in tqdm(sample_files, desc="Reconstructing molecules"):
    target_geom = read(os.path.join(target_folder, target_file))
    target_soap = generate_SOAP(target_geom)

    current_geom = ref_geom.copy().get_positions()
    trajectory = []

    for step in range(max_steps):
        grad = finite_gradient(current_geom, target_soap, eps=eps)
        current_geom -= lr * grad
        trajectory.append(current_geom.copy())

    # Save trajectory
    traj_atoms_list = [Atoms(positions=pos, symbols=target_symbols) for pos in trajectory]
    traj_path = os.path.join(output_folder, f"{target_file[:-4]}_trajectory.xyz")
    write(traj_path, traj_atoms_list)

    # Save final reconstructed geometry
    reconstructed = Atoms(positions=current_geom, symbols=target_symbols)
    write(os.path.join(output_folder, f"{target_file[:-4]}_reconstructed.xyz"), reconstructed)

Reconstructing molecules:   0%|          | 0/5 [05:38<?, ?it/s]
/opt/homebrew/anaconda3/envs/conf-align/lib/python3.12/site-packages/dscribe/descriptors/soap.py:71: SyntaxWarning: invalid escape sequence '\s'
  """


KeyboardInterrupt: 